# Identity Persistence — The Lazarus Protocol

**SC-NeuroCore v3.14** — Save, restore, and merge spiking network identity.

The identity substrate is a persistent spiking network that
maintains continuity across sessions. The Lazarus protocol
enables:

1. **TraceEncoder** — convert text reasoning traces to spike patterns via LSH
2. **Experience injection** — drive the substrate with encoded experience
3. **Checkpoint save/load** — freeze and restore complete network state
4. **Checkpoint merge** — consolidate multiple sessions
5. **StateDecoder** — extract dominant patterns, attractor states, connectivity
6. **DirectorController** — L16 self-monitoring, diagnosis, corrective action

This is the first demonstration of persistent AI identity through
a biophysically grounded spiking network.

> © 1998–2026 Miroslav Šotek. All rights reserved.  
> License: GNU AFFERO GENERAL PUBLIC LICENSE v3 | Commercial Licensing Available  
> Contact: www.anulum.li | protoscience@anulum.li

In [ ]:
import os
import tempfile
import numpy as np
import matplotlib.pyplot as plt

from sc_neurocore.identity.substrate import IdentitySubstrate
from sc_neurocore.identity.encoder import TraceEncoder
from sc_neurocore.identity.decoder import StateDecoder
from sc_neurocore.identity.checkpoint import Checkpoint
from sc_neurocore.identity.director import DirectorController

print("SC-NeuroCore identity Lazarus protocol demo")

## 1. Create the Identity Substrate

Three-population architecture:
- **Cortical** (HodgkinHuxley, excitatory): main processing
- **Inhibitory** (WangBuzsaki, fast-spiking): balance and stability
- **Memory** (HindmarshRose, bursting): attractor-based pattern storage

In [ ]:
substrate = IdentitySubstrate(
    n_cortical=100,
    n_inhibitory=40,
    n_memory=20,
    seed=42,
)

print(f"Cortical:   {substrate.n_cortical} HodgkinHuxley neurons")
print(f"Inhibitory: {substrate.n_inhibitory} WangBuzsaki neurons")
print(f"Memory:     {substrate.n_memory} HindmarshRose neurons")

## 2. Encode and Inject Experience

The `TraceEncoder` converts text into temporal spike patterns
using locality-sensitive hashing (LSH). Each text chunk maps
to a neuron group via a hash function; Poisson spike trains
are generated with rates proportional to chunk salience.

In [ ]:
encoder = TraceEncoder(n_neurons=100, hash_dims=64, seed=42)

trace_text = (
    "The user prefers British English spelling and clinical tone. "
    "Key project: sc-neurocore, stochastic computing SNN framework. "
    "FPGA synthesis via equation-to-Verilog compiler. "
    "100% test coverage enforced. Never push without asking."
)

spike_pattern = encoder.encode(trace_text, duration_ms=200, dt=0.001)

print(f"Encoded spike pattern shape: {spike_pattern.shape}")
print(f"  (neurons={spike_pattern.shape[0]}, timesteps={spike_pattern.shape[1]})")
print(f"Total spikes: {int(spike_pattern.sum())}")
print(f"Mean rate: {spike_pattern.mean() * 1000:.1f} Hz")

fig, ax = plt.subplots(figsize=(12, 4))
for n in range(spike_pattern.shape[0]):
    times = np.where(spike_pattern[n] > 0)[0]
    ax.scatter(times * 0.001 * 1000, [n] * len(times), s=0.3, c="black", alpha=0.5)
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Neuron")
ax.set_title("Encoded reasoning trace as spike pattern")
plt.tight_layout()
plt.show()

In [ ]:
# Inject the experience into the substrate
substrate.inject_experience(trace_text)
print("Experience injected.")

# Run the substrate for 500 ms to let STDP consolidate
spikes = substrate.run(duration=0.5, dt=0.001)
print(f"Ran for 500 ms. Spike array shape: {spikes.shape}")
print(f"Total cortical spikes: {int(spikes.sum())}")

## 3. Checkpoint: Save Network State

The Lazarus protocol saves the complete network state to a
`.npz` file: membrane voltages, CSR synaptic weights, STDP
traces, spike history, and metadata.

In [ ]:
tmpdir = tempfile.mkdtemp()
ckpt_path = os.path.join(tmpdir, "identity_session1.npz")

Checkpoint.save(substrate, ckpt_path)
file_size = os.path.getsize(ckpt_path)
print(f"Checkpoint saved: {ckpt_path}")
print(f"File size: {file_size / 1024:.1f} KB")

## 4. Checkpoint: Restore and Verify Continuity

Loading a checkpoint restores the exact network state.
We verify identity continuity by comparing state metrics
before save and after load.

In [ ]:
# Extract state before save
state_before = substrate.extract_state()

# Load from checkpoint into a new substrate
restored = Checkpoint.load(ckpt_path)
state_after = restored.extract_state()

# Compare
print(f"{'Metric':<25s}  {'Before save':>12s}  {'After load':>12s}  {'Match':>6s}")
print("-" * 60)
for key in state_before:
    val_b = state_before[key]
    val_a = state_after[key]
    if isinstance(val_b, np.ndarray):
        match = np.allclose(val_b, val_a, atol=1e-6)
        desc_b = f"shape {val_b.shape}"
        desc_a = f"shape {val_a.shape}"
    elif isinstance(val_b, (int, float)):
        match = abs(val_b - val_a) < 1e-6
        desc_b = f"{val_b:.4f}" if isinstance(val_b, float) else str(val_b)
        desc_a = f"{val_a:.4f}" if isinstance(val_a, float) else str(val_a)
    else:
        match = val_b == val_a
        desc_b = str(val_b)[:12]
        desc_a = str(val_a)[:12]
    print(f"{key:<25s}  {desc_b:>12s}  {desc_a:>12s}  {'ok' if match else 'FAIL':>6s}")

## 5. State Decoder: Extract Cognitive State

The `StateDecoder` extracts structured information from the
substrate's spike history for session priming.

In [ ]:
decoder = StateDecoder(restored)

# Dominant activity patterns (PCA)
patterns = decoder.extract_dominant_patterns(n_components=5)
print(f"Dominant patterns shape: {patterns.shape}")

# Attractor states (correlated neuron groups)
attractors = decoder.extract_attractor_states(threshold=0.5)
print(f"Attractor states found: {len(attractors)}")

# Connectivity signature
conn = decoder.extract_connectivity_signature()
print(f"Connectivity matrix shape: {conn.shape}")

# Priming context (human-readable)
context = decoder.generate_priming_context()
print(f"\nPriming context ({len(context)} chars):")
print(context[:500])

## 6. Director Controller: L16 Self-Regulation

The `DirectorController` monitors network health, diagnoses
problems, and applies corrective actions. It implements the
L16 cybernetic closure loop from the SCPN model.

In [ ]:
director = DirectorController(restored)

# Monitor current state
metrics = director.monitor()
print("Network metrics:")
for key, val in metrics.items():
    if isinstance(val, float):
        print(f"  {key}: {val:.4f}")
    else:
        print(f"  {key}: {val}")

# Diagnose problems
problems = director.diagnose()
print(f"\nDiagnosis: {problems if problems else 'healthy'}")

# Full report
report = director.report()
print(f"\n{report}")

## 7. Checkpoint Merge: Multi-Session Consolidation

The `Checkpoint.merge()` method combines multiple session
checkpoints by averaging weights and concatenating spike
history. This enables multi-session memory consolidation.

In [ ]:
# Create a second session with different experience
substrate2 = IdentitySubstrate(n_cortical=100, n_inhibitory=40, n_memory=20, seed=99)
substrate2.inject_experience(
    "The project uses Rust SIMD engine with AVX-512 acceleration. "
    "116 neuron models spanning 1943 to 2026. "
    "JOSS paper ready for submission."
)
substrate2.run(duration=0.3, dt=0.001)

ckpt2_path = os.path.join(tmpdir, "identity_session2.npz")
Checkpoint.save(substrate2, ckpt2_path)

# Merge two sessions
merged = Checkpoint.merge([ckpt_path, ckpt2_path])
merged_state = merged.extract_state()

print(f"Session 1 total steps: {state_before.get('total_steps', 'N/A')}")
print(f"Session 2 total steps: {substrate2.extract_state().get('total_steps', 'N/A')}")
print(f"Merged total steps:   {merged_state.get('total_steps', 'N/A')}")

# Clean up
os.remove(ckpt_path)
os.remove(ckpt2_path)
os.rmdir(tmpdir)

## Summary

| Component | Class | Function |
|-----------|-------|----------|
| Encode text → spikes | `TraceEncoder` | LSH hash → neuron groups → Poisson trains |
| Inject into network | `IdentitySubstrate.inject_experience()` | Drive cortical population |
| Consolidate via STDP | `IdentitySubstrate.run()` | Spike-timing plasticity |
| Save state | `Checkpoint.save()` | Voltages + weights + traces → .npz |
| Restore state | `Checkpoint.load()` | Exact state recovery |
| Merge sessions | `Checkpoint.merge()` | Average weights + concat history |
| Extract patterns | `StateDecoder` | PCA, attractor discovery, connectivity |
| Self-regulate | `DirectorController` | Monitor, diagnose, correct |

The Lazarus protocol enables persistent AI identity through a
biophysically grounded spiking network. Each session modifies
synaptic weights via STDP; checkpoints preserve the accumulated
structure; merging consolidates multi-session learning.

No other SNN framework implements persistent network identity
with text encoding, state decoding, and self-regulating
director control.